In [1]:
import hyperopt

print(hyperopt.__version__)

0.2.7


In [2]:
from hyperopt import hp

search_space = {"x": hp.quniform("x", -10, 10, 1),
                "y": hp.quniform("y", -15, 15, 1)}


In [3]:
from hyperopt import STATUS_OK

def objective_func(search_space):
    x = search_space["x"]
    y = search_space["y"]
    retval = x**2 - 20*y
    return retval

In [4]:
from hyperopt import fmin, tpe, Trials
import numpy as np

trial_val = Trials()

best_01 = fmin(fn = objective_func, space = search_space,
                algo = tpe.suggest, max_evals = 5, trials = trial_val,
                rstate = np.random.default_rng(seed = 0))
print(best_01)

100%|██████████| 5/5 [00:00<00:00, 1000.74trial/s, best loss: -224.0]


{'x': -4.0, 'y': 12.0}


In [5]:
trial_val = Trials()

best_02 = fmin(fn = objective_func, space = search_space,
                algo = tpe.suggest, max_evals = 20, trials = trial_val,
                rstate = np.random.default_rng(seed = 0))
print(best_02)

100%|██████████| 20/20 [00:00<00:00, 1537.67trial/s, best loss: -296.0]
{'x': 2.0, 'y': 15.0}


In [6]:
print(trial_val.results)

[{'loss': -64.0, 'status': 'ok'}, {'loss': -184.0, 'status': 'ok'}, {'loss': 56.0, 'status': 'ok'}, {'loss': -224.0, 'status': 'ok'}, {'loss': 61.0, 'status': 'ok'}, {'loss': -296.0, 'status': 'ok'}, {'loss': -40.0, 'status': 'ok'}, {'loss': 281.0, 'status': 'ok'}, {'loss': 64.0, 'status': 'ok'}, {'loss': 100.0, 'status': 'ok'}, {'loss': 60.0, 'status': 'ok'}, {'loss': -39.0, 'status': 'ok'}, {'loss': 1.0, 'status': 'ok'}, {'loss': -164.0, 'status': 'ok'}, {'loss': 21.0, 'status': 'ok'}, {'loss': -56.0, 'status': 'ok'}, {'loss': 284.0, 'status': 'ok'}, {'loss': 176.0, 'status': 'ok'}, {'loss': -171.0, 'status': 'ok'}, {'loss': 0.0, 'status': 'ok'}]


In [7]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

dataset = load_breast_cancer()
df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)
df["target"] = dataset.target

X_featurees = df.iloc[:, :-1]
y_label = df.iloc[:, -1]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_featurees, y_label, test_size=0.2, random_state=156)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=156)

In [17]:
from hyperopt import hp

xgb_search_space = {
    "max_depth": hp.quniform("max_depth", 5, 20, 1),
    "min_child_weight": hp.quniform("min_child_weight", 1, 2, 1),
    "learning_rate": hp.uniform("learning_rate", 0.01, 0.2),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.5, 1)
}

In [18]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from hyperopt import STATUS_OK

def objective_func(search_space):
    xgb_clf = XGBClassifier(n_estimators = 100,
                            max_depth = int(search_space["max_depth"]),
                            min_child_weight = int(search_space["min_child_weight"]),
                            learning_rate = int(search_space["colsample_bytree"]),
                            eval_metric = "logloss")
    accuracy = cross_val_score(xgb_clf, X_train, y_train,
                           scoring = "accuracy", cv = 3)
    return {"loss": -1 * np.mean(accuracy), "status": STATUS_OK}

In [19]:
from hyperopt import fmin, tpe, Trials

trial_val = Trials()
best = fmin(fn = objective_func, space = xgb_search_space,
            algo = tpe.suggest, max_evals = 50,
            trials = trial_val, rstate = np.random.default_rng(seed = 9))
print(best)

100%|██████████| 50/50 [00:10<00:00,  4.81trial/s, best loss: -0.3846142674567213]
{'colsample_bytree': 0.5852347138193622, 'learning_rate': 0.033687779759699644, 'max_depth': 19.0, 'min_child_weight': 2.0}


In [20]:
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score

def get_clf_eval(y_test, pred = None, pred_proba = None):
    confusion = confusion_matrix(y_test, pred)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    roc_auc = roc_auc_score(y_test, pred_proba)
    print("오차행렬")
    print(confusion)
    print("정확도: ", accuracy)
    print("정밀도: ", precision)
    print("재현율: ", recall)
    print("f1: ", f1)
    print("roc_auc: ", roc_auc)

In [21]:
xgb_wrapper = XGBClassifier(n_estimators = 400,
                            learning_rate = round(best["learning_rate"], 5),
                            max_depth = int(best["max_depth"]),
                            min_child_weight = int(best["min_child_weight"]),
                            colsample_bytree = round(best["colsample_bytree"], 5))

evals = [(X_tr, y_tr), (X_val, y_val)]
xgb_wrapper.fit(X_tr, y_tr, early_stopping_rounds = 50,
                eval_metric = "logloss", eval_set = evals, verbose = True)
preds = xgb_wrapper.predict(X_test)
pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

get_clf_eval(y_test, preds, pred_proba)

[0]	validation_0-logloss:0.66460	validation_1-logloss:0.66902
[1]	validation_0-logloss:0.63717	validation_1-logloss:0.64749
[2]	validation_0-logloss:0.61177	validation_1-logloss:0.62491
[3]	validation_0-logloss:0.58746	validation_1-logloss:0.60568
[4]	validation_0-logloss:0.56525	validation_1-logloss:0.58838
[5]	validation_0-logloss:0.54361	validation_1-logloss:0.57055
[6]	validation_0-logloss:0.52347	validation_1-logloss:0.55277
[7]	validation_0-logloss:0.50466	validation_1-logloss:0.53890
[8]	validation_0-logloss:0.48656	validation_1-logloss:0.52473
[9]	validation_0-logloss:0.46917	validation_1-logloss:0.50979
[10]	validation_0-logloss:0.45261	validation_1-logloss:0.49609
[11]	validation_0-logloss:0.43710	validation_1-logloss:0.48397
[12]	validation_0-logloss:0.42238	validation_1-logloss:0.47216
[13]	validation_0-logloss:0.40829	validation_1-logloss:0.46179
[14]	validation_0-logloss:0.39455	validation_1-logloss:0.45064
[15]	validation_0-logloss:0.38150	validation_1-logloss:0.43999
[1

c:\Users\Win11Pro\Desktop\Learning\MachineLearing\.venv\lib\site-packages\xgboost\sklearn.py:835: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
c:\Users\Win11Pro\Desktop\Learning\MachineLearing\.venv\lib\site-packages\xgboost\sklearn.py:835: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[105]	validation_0-logloss:0.05113	validation_1-logloss:0.22119
[106]	validation_0-logloss:0.05051	validation_1-logloss:0.22114
[107]	validation_0-logloss:0.04987	validation_1-logloss:0.22054
[108]	validation_0-logloss:0.04924	validation_1-logloss:0.22084
[109]	validation_0-logloss:0.04856	validation_1-logloss:0.22027
[110]	validation_0-logloss:0.04795	validation_1-logloss:0.22022
[111]	validation_0-logloss:0.04744	validation_1-logloss:0.22056
[112]	validation_0-logloss:0.04685	validation_1-logloss:0.22087
[113]	validation_0-logloss:0.04621	validation_1-logloss:0.22092
[114]	validation_0-logloss:0.04567	validation_1-logloss:0.22128
[115]	validation_0-logloss:0.04513	validation_1-logloss:0.22049
[116]	validation_0-logloss:0.04455	validation_1-logloss:0.22017
[117]	validation_0-logloss:0.04408	validation_1-logloss:0.21985
[118]	validation_0-logloss:0.04345	validation_1-logloss:0.21914
[119]	validation_0-logloss:0.04291	validation_1-logloss:0.21874
[120]	validation_0-logloss:0.04242	valid